# MapReduce-Reloaded: the demo, end to end

*A from-scratch implementation of Google's MapReduce (Dean & Ghemawat, OSDI
2004), run live on one coordinator and four one-CPU workers. Each live result
below is produced and checked as the notebook runs, and a scorecard at the
end gathers what held; the evaluation figures at the end are earlier
measurements.*

This notebook runs the demo in [`docs/demo.md`](../docs/demo.md) against the
Compose cluster, one claim per section. Sections 7, 8, 10, 11 and 13 check
their claims with `assert`, and 13 checks section 5's output byte for byte;
section 9 checks that its job succeeds, not its counts. The others show
what the cluster did, and any command that fails stops the run:

| § | What it shows | Paper |
|---|---|---|
| 2–5 | Four one-CPU workers; upload, submit and read word count from the host | §2, §3.1 |
| 6 | The answer, plotted | |
| 7 | A job file: map and reduce as Go source, interpreted by the workers | §2 |
| 8 | The combiner: fewer bytes shuffled, the same output | §4.3 |
| 9 | A second program, URL access frequency | §2.3 |
| 10 | A worker killed mid-job: its tasks run again elsewhere | §3.3 |
| 11 | The coordinator restarted mid-job: the job resumes from its log | beyond the paper |
| 12 | The coordinator's metrics | §4.8, §4.9 |
| 13 | Local mode agrees with the cluster byte for byte | §4.7 |
| 14 | The evaluation at a glance: the measured results | §5 |
| 15 | If a cell fails: what to check and run | |

**How the system fits together.** The `mr` client on the host submits a
job to the coordinator, the paper's master, over gRPC. Four workers, each a
container limited to one CPU, pull map and reduce tasks from the
coordinator, send it heartbeats, and fetch each other's map output directly
for the shuffle, so no data passes through the coordinator. Input and output
live in the host's `data/` directory, which every container sees at
`/data`; the coordinator writes every decision to a write-ahead log, so it
can crash and resume. [`design.md`](../design.md#components) has the details.

```text
mr (host) ──submit, status──▶ coordinator ◀──pull tasks, heartbeats── workers 1–4
                              (write-ahead log)                       ⇅ shuffle, worker to worker
          input and output: data/ on the host, /data in every container
```

The text before each section's code tells the story as it goes: what the
cell does, what to point at in its output, and how it leads to the next.

**Before you start**, on the VM, from the repository root:

```sh
make cluster-up mr      # the cluster and the client
make upload-example     # the corpus and a 10 MB web log
python/.venv/bin/pip install -e 'python[notebook]'   # Jupyter and matplotlib
```

Then open it with `python/.venv/bin/jupyter notebook examples/demo.ipynb`, or
run every cell without a browser:

```sh
python/.venv/bin/jupyter nbconvert --to notebook --execute --inplace examples/demo.ipynb
```

*Kernel → Restart & Run All* takes about two minutes on the course VM, almost
all of it in sections 10 and 11, which sort 1 GiB each (the input is
generated on first use: about 20 seconds and 1 GB of disk). Set
`KILL_WORKER = False` or `RESTART_COORDINATOR = False` in the setup cell to
skip either. Rerunning from the top is safe once the previous run has
finished or stopped: each run uses timestamped output directories, the last
cell removes them, and the setup cell removes any an interrupted run left.
Run one copy at a time against a cluster: the setup cell would also remove
the directories of a copy still running. The status page is at
<http://127.0.0.1:18780/>; keep it open beside the notebook. If a cell
fails, section 15 says what to check.


### How to open this notebook

Everything runs from the prepared checkout, `~/mr-checkout` on the course
VM: launching Jupyter elsewhere leaves §1's `find_root()` pointing at the
wrong tree — and §1 refuses a bare or out-of-date checkout outright, so a
stale clone fails fast with the reason instead of surprising you later. On the VM:

```sh
cd ~/mr-checkout
make cluster-up mr && make upload-example
python/.venv/bin/pip install -e 'python[notebook]'
python/.venv/bin/jupyter notebook --no-browser --port=8888
```

On the laptop, open the same tunnels as [the runbook](../docs/runbook.md)
— Jupyter (8888), the status page (18780) and Grafana (13000) — and follow
the token URL Jupyter prints, which carries its login token:

```sh
ssh -L 8888:127.0.0.1:8888 -L 18780:127.0.0.1:18780 -L 13000:127.0.0.1:13000 mr-vm
```


---

### What this demo shows

Google's 2004 MapReduce paper made a promise: write two small functions,
`Map` and `Reduce`, and the system takes care of everything else. It splits
the input, spreads the work over many machines, moves the intermediate data
between them, and survives the machines that fail along the way. This
notebook puts that promise to the test on our own implementation, live, on
a coordinator and four workers, each worker a container limited to one CPU.

We start with the model itself: upload a few books, count their words on
the cluster and read the answer back. Then come two things built on top of
the basics: a job written as a plain Go file that the workers interpret,
and the combiner, which cuts the data sent across the network. Then the part
that matters most in a distributed system. We kill a worker in the middle of
a 1 GiB sort, and then crash the coordinator itself in the middle of
another, which is the failure the 2004 paper simply gives up on. We finish
with the evidence: the coordinator's own record of every job, a byte-for-byte
check against a single-process run, and the measured results.

None of this is a screenshot. The jobs run live, and the claims that matter
are checked by an `assert` as they run: the job file's output, the
combiner's saving, the lost worker, the coordinator restart and the
byte-for-byte check. If we reach the end, those claims held on this cluster
today. Two things are not checked here. A fault section switched off in the
setup cell (`KILL_WORKER` or `RESTART_COORDINATOR`) is skipped, so its claim
is not tested on this run. And the evaluation figures at the end are earlier
measurements, shown rather than rerun, since they took about two hours.


---

## 1. Setup


Before any job runs, the first cell gets the ground under our feet. It
defines the small helpers the rest of the notebook leans on: `run()` calls a
command from the repository root, `submit()` sends a job and waits for it,
and `cat()` reads a job's output back. It also stamps this run with the
time, so every output directory is new and nothing collides with an earlier
rehearsal.

Then it makes sure we are demonstrating the real thing: a git checkout on
`main` with nothing from GitHub missing, the `mr` client built, a coordinator
answering on port 18780, and workers alive. If any of that is wrong, it
stops here and says why, instead of failing halfway through the demo. It also
clears away anything an interrupted earlier run left behind. All we should
see is one line saying the checkout is ready and that four workers are live.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

from IPython.display import Image, Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("retina")  # sharp figures on a projector or a high-DPI screen

# Sections 10 and 11 each sort 1 GiB: one kills a worker, the other
# restarts the coordinator. False skips a section.
KILL_WORKER = True
RESTART_COORDINATOR = True


def find_root():
    """The repository root: MR_ROOT, or the first of the kernel's working
    directory and its parents that holds deploy/compose. Jupyter starts the
    kernel in examples/, the notebook's directory."""
    if "MR_ROOT" in os.environ:
        return Path(os.environ["MR_ROOT"])
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "deploy" / "compose" / "compose.yaml").exists():
            return path
    raise RuntimeError("not inside the repository: set MR_ROOT to its path")


ROOT = find_root()
MR = ROOT / "bin" / "mr"
RUN = f"{int(time.time())}-{os.getpid()}"  # unique per run, even within a second
CORPUS = f"/data/input/notebook-corpus-{RUN}"
COMPOSE = ["docker", "compose", "-p", "mapreduce", "-f", "deploy/compose/compose.yaml"]
METRICS = "http://127.0.0.1:18780/metrics"
STATUS = "http://127.0.0.1:18780/status.json"  # the status page, as data
EXPECTED = b"".join(p.read_bytes() for p in sorted((ROOT / "testdata/corpus/expected").glob("part-r-*")))
JOBS = {}  # this run's job ids and what each ran, for section 12
CHECKS = {}  # each claim a section checked, by section and claim, for the scorecard
# A run stopped by a failing cell never reaches the last section's cleanup,
# so clear what earlier runs left before this one adds more.
for old in [*(ROOT / "data/out").glob("notebook-*"), *(ROOT / "data/input").glob("notebook-corpus-*")]:
    shutil.rmtree(old, ignore_errors=True)
SORT_INPUT = ROOT / "data/input/sort-1g.txt"
# The project's chart colors (bench/plot-experiments.py): blue is the
# system as shipped, orange the changed condition, text in ink, not color.
BLUE, ORANGE, INK, INK2 = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"


def run(*args, check=True, timeout=None):
    """Runs a command in the repository root and returns the result. A
    command that outlasts timeout seconds raises subprocess.TimeoutExpired."""
    result = subprocess.run(
        [str(a) for a in args], cwd=ROOT, text=True, capture_output=True, timeout=timeout, check=False
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, args))} failed:\n{result.stderr}")
    return result


def submit(label, *args):
    """Submits a job with -wait, records its id under label for section 12,
    and returns its status reply."""
    reply = run(MR, "submit", "-wait", *args).stdout
    job, state = reply.splitlines()[0].split("\t")
    assert state == "SUCCEEDED", reply
    JOBS[job] = label
    return reply


def cat(path):
    """A job's output, its part files in order, as bytes."""
    return subprocess.run([MR, "cat", path], cwd=ROOT, capture_output=True, check=True).stdout


def digest(path):
    """A SHA-256 of a job's part files, in order, read from the host's data/
    directory in chunks (a 1 GiB output does not fit a notebook's memory)."""
    h = hashlib.sha256()
    for part in sorted((ROOT / path.removeprefix("/")).glob("part-r-*")):
        with part.open("rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
    return h.hexdigest()


SORT_BYTES = -(-(2**30) // 100) * 100  # bench/gen-sort.py writes whole 100-byte records


def sort_input():
    """Generates the 1 GiB sort input on first use. It is written to a
    temporary file and renamed when complete, and a file of the wrong size,
    such as one left by an interrupted run, is generated again."""
    if SORT_INPUT.exists() and SORT_INPUT.stat().st_size == SORT_BYTES:
        return
    free = shutil.disk_usage(ROOT / "data").free
    assert free > 3 * 2**30, f"{free / 2**30:.1f} GiB free: the sort needs 3 GiB"
    partial = SORT_INPUT.with_name(SORT_INPUT.name + ".partial")
    with partial.open("wb") as out:
        subprocess.run(["python3", "bench/gen-sort.py", str(2**30), "42"], cwd=ROOT, stdout=out, check=True)
    assert partial.stat().st_size == SORT_BYTES, f"{partial} is {partial.stat().st_size} bytes, not {SORT_BYTES}"
    partial.replace(SORT_INPUT)


def live_workers():
    """The number of workers the coordinator counts as live, or 0."""
    try:
        return next((int(line.split()[1]) for line in metrics_lines() if line.startswith("mr_workers ")), 0)
    except (urllib.error.URLError, OSError):
        return 0


def wait_for_workers(n=4, seconds=60):
    """Waits until n workers are live, and returns whether they are."""
    for _ in range(seconds):
        if live_workers() >= n:
            return True
        time.sleep(1)
    return False


def style(axis, title):
    """The project's chart style: a left-aligned finding as the title,
    hairline axes, no box."""
    axis.set_title(title, loc="left", fontsize=11, color=INK)
    for side in ("top", "right"):
        axis.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        axis.spines[side].set_color("#c3c2b7")
    axis.tick_params(colors="#c3c2b7", labelcolor=INK2)


def status_json():
    """The coordinator's status page as data: its jobs and live workers."""
    with urllib.request.urlopen(STATUS, timeout=5) as response:
        return json.load(response)


def containers():
    """Each running container's name, by its short ID, which is also the
    hostname a worker registers with."""
    lines = run("docker", "ps", "--format", "{{.ID}} {{.Names}}").stdout.splitlines()
    return dict(line.split(" ", 1) for line in lines)


def table(header, rows):
    """Shows rows as a Markdown table, which Jupyter renders."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def metrics_lines():
    """The coordinator's /metrics, as text lines."""
    with urllib.request.urlopen(METRICS, timeout=5) as response:
        return response.read().decode().splitlines()


def fields(text):
    """A status reply, as a dict: job-1\tSUCCEEDED, map\t4/4, counters."""
    return dict(line.split("\t", 1) for line in text.splitlines() if "\t" in line)


def outcome(reply):
    """A status reply's headline on one line: job, state, tasks and time."""
    status = fields(reply)
    job, state = reply.splitlines()[0].split("\t")
    return f"{job} {state} · map {status['map']} · reduce {status['reduce']} · {float(status['seconds']):.1f} s"


def check_key(section, claim):
    """A claim's scorecard key: its section and its words before any colon,
    which hold measured values that change between runs. Running a section
    again replaces its row instead of adding a second one."""
    return section, claim.split(":", 1)[0]


def passed(section, claim):
    """Records a claim whose asserts just held, for the scorecard, and says so."""
    CHECKS[check_key(section, claim)] = (claim, "✓ held")
    print(f"✓ {claim}")


def skipped(section, claim, reason):
    """Records a claim whose section was switched off: not tested on this run."""
    CHECKS[check_key(section, claim)] = (claim, f"skipped ({reason}): not tested")
    print(f"skipped ({reason}): {claim}")


def caption(number, text):
    """Captions the figure just shown, numbered in the order they appear."""
    display(Markdown(f"**Figure {number}.** {text}"))


# The demo runs from a work tree on main that is current with origin/main.
# A bare or stale copy (an old clone that lost its work tree) passes
# find_root()'s file test and would run against the wrong tree, so say so.
status = run("git", "-C", ROOT, "rev-parse", "--is-inside-work-tree", check=False).stdout.strip()
assert status == "true", (
    f"{ROOT} is not a git work tree (a bare copy cannot run the demo);"
    " clone the repository and launch this notebook from its examples/ directory")
branch = run("git", "-C", ROOT, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
assert branch == "main", f"{ROOT} is on {branch}; the demo runs from main"
try:  # a network that hangs fails here in a minute, not never
    run("git", "-C", ROOT, "fetch", "origin", "main", timeout=60)
except (RuntimeError, subprocess.TimeoutExpired) as error:
    raise RuntimeError(
        f"could not refresh origin/main ({error}); a stale checkout cannot"
        " be verified current") from error
behind = run("git", "-C", ROOT, "rev-list", "--count", "HEAD..origin/main").stdout.strip()
assert behind == "0", (
    f"{ROOT} is {behind} commit(s) behind origin/main;"
    " run `git pull`, then restart the kernel and run again")
assert MR.exists(), "bin/mr is missing: run `make cluster-up mr` first"
try:
    metrics_lines()
except (urllib.error.URLError, OSError) as error:
    raise RuntimeError(f"the coordinator is not answering at {METRICS}: run `make cluster-up`") from error
# A job waits forever for a worker, so stop here if none is live.
workers = live_workers()
if workers == 0:
    raise RuntimeError("the coordinator has no live workers: run `make cluster-up`")
print(f"ready: {ROOT}, on main and up to date with origin/main; live workers: {workers}")


---

## 2. Check the cluster

One coordinator and four workers, each limited to one CPU. Docker lists the
containers; the coordinator's own view, from its status page, lists the
workers it has registered under its own IDs, which container each is, and
how long ago each last sent a heartbeat. A
worker silent for 10 seconds is declared lost (section 10).


This is the whole cluster: one coordinator, which is the paper's master, and
four workers. The first table is Docker's list of the containers, with
Prometheus beside them when the monitoring is up. Below it is the
coordinator's own view, read from its status page, of the workers it has
registered. The two should agree.

The ids are the coordinator's names for its workers. `r1` means this is the
coordinator's first life since the cluster was last cleaned, and `w1` to
`w4` follow the order in which the workers happened to register, which is
why they don't line up with the container numbers. Each worker has one task
slot because it has one CPU. Workers send a heartbeat every three seconds,
so the last heartbeat should be fresh; a worker silent for ten seconds is
declared lost, and we will make exactly that happen in section 10.

One design point is worth saying out loud: the coordinator only schedules.
Map output stays on the worker that produced it, and reducers fetch it from
that worker directly, so no data ever flows through the coordinator. The
link below opens the live status page. Keep it beside the notebook from here
on, because every job we run shows up there as it runs.


In [ ]:
ps = run(*COMPOSE, "ps", "--format", "{{.Name}}\t{{.Status}}").stdout
if ps:
    table(["Container", "Docker status"], sorted(line.split("\t", 1) for line in ps.splitlines()))
else:
    print("no containers: run `make cluster-up mr`")
print(f"The coordinator counts {live_workers()} live workers:")

status, names = status_json(), containers()
table(
    ["Worker", "Container", "Task slots", "Last heartbeat", "Running"],
    [
        (w["id"], names.get(w["host"], w["host"]), w["slots"], f'{status["at"] - w["last_seen"]:.1f} s ago', len(w["running"]))
        for w in sorted(status["workers"], key=lambda w: w["id"])
    ],
)

In [ ]:
# The workers' live view: the status page refreshes itself every few
# seconds, so the heartbeats, running tasks and progress bars stay current
# through the whole run. On the laptop the same URL rides the SSH forward.
display(Markdown("[Open the status page](http://127.0.0.1:18780/): workers' heartbeats, running tasks and every job's progress bars, refreshed every few seconds."))

---

## 3. Upload the corpus

Every container sees the host's `data/` directory at `/data`.


Now some data. The `mr` client runs on the host, not in a container, and
`mr upload` copies the four public-domain books in `testdata/corpus/`, about
half a megabyte of text, into the shared `data/` directory. Every container
sees that directory at `/data`, so a path like `/data/input/...` means the
same file to the client, the coordinator and all four workers. It plays the
part GFS played in the paper (ADR 0016). The four file names come back to
confirm the upload.


In [ ]:
run(MR, "upload", "testdata/corpus/*.txt", CORPUS)
print(sorted(p.name for p in (ROOT / CORPUS.removeprefix("/")).iterdir()))


---

## 4. Submit word count

The compiled `wordcount` program, by name: 4 map tasks and 4 reduce tasks.
`-wait` polls the job's status until it ends, and the reply ends with the
job's counters, the paper's §4.9: records and bytes read by the map tasks,
emitted by them, folded by the combiner, and written by the reduce tasks.


Here is the paper's programming model at work. We submit the built-in
`wordcount` program with 4 map tasks and 4 reduce tasks. The coordinator
cuts the input into four splits and hands them out as workers ask for work.
Each map task reads its split, emits a `(word, 1)` pair for every word,
sorts and combines those pairs, and writes one file per reduce task to its
own worker's disk. Each reduce task then pulls its share from every map
worker, merges them, adds up the counts and writes one output file.

The counters in the reply tell that story in numbers, and the line under
it sums the job up. The map
tasks read 9,316 lines and emitted 78,658 words. The combiner folded those
down to 19,333 before anything crossed the network, and the reducers wrote
16,851 distinct words.

Two lines may draw questions. `backups` can be above zero even on a job this
small, and it changes from run to run. With as many reduce tasks as workers,
the first worker to finish has nothing left to do, so the coordinator hands
it a backup copy of a task that is still running (the paper's §3.6), and
`backups_won` counts the copies that finished first. They use workers that
would otherwise sit idle, so they cost nothing. `map_host_local` is 0
because locality only matters when input lives on particular machines, and
here every worker reads from the same shared directory.


In [ ]:
WC = f"/data/out/notebook-wc-{RUN}"
reply = submit("word count, compiled", "-m", "4", "-r", "4", "wordcount", WC, f"{CORPUS}/*.txt")
print(reply)
print(outcome(reply))
passed(4, "Word count ran on the cluster and succeeded")


---

## 5. Read the output


Reading the answer back takes one command: `mr cat` joins the four output
files in order. Within each file the words come out sorted, as the paper's
§4.2 promises; ranking the ten most frequent happens here in Python, after
the job, and the table gives each one's share of the whole. There are 16,851 distinct words, and "the" leads with 3,444.


In [ ]:
output = cat(WC)
words = [line.split("\t") for line in output.decode().splitlines()]
total = sum(int(count) for _, count in words)
top = sorted(((word, int(count)) for word, count in words), key=lambda pair: pair[1], reverse=True)[:10]
print(f"{len(words):,} distinct words, {total:,} words in all. The ten most frequent:")
table(["Rank", "Word", "Count", "Share of all words"], [(rank, word, f"{count:,}", f"{count / total:.1%}") for rank, (word, count) in enumerate(top, 1)])


---

## 6. Plot the ten most frequent words


Figure 1 is the same answer as a picture. "The" alone is 4.4% of all the words in the
corpus, and the familiar small words follow it down the list.


In [ ]:
import matplotlib.pyplot as plt

words_top, counts_top = [w for w, _ in reversed(top)], [c for _, c in reversed(top)]
figure, axis = plt.subplots(figsize=(7, 3.6))
axis.barh(words_top, counts_top, color=BLUE, height=0.6)
for y, count in enumerate(counts_top):
    axis.annotate(f"{count:,}", (count, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Occurrences in the test corpus", color=INK2)
axis.set_xlim(0, counts_top[-1] * 1.12)
axis.grid(axis="x", color="#e1e0d9", linewidth=1)
axis.set_axisbelow(True)
style(axis, f'"{top[0][0]}" alone is {top[0][1] / total:.1%} of all words')
figure.tight_layout()
plt.show()
caption(1, f"The ten most frequent of the corpus's {len(words):,} distinct words, counted by the cluster in §4 and read back in §5. The short function words of English lead, as they do in any large body of text.")


---

## 7. The same job as a job file

`examples/wordcount.go` is word count as Go source. It is sent with the job
and interpreted by the workers (ADR 0015): nothing is compiled or uploaded.
Its output must be the compiled program's, byte for byte.


Now the first thing built beyond the paper's basics. The assignment's
client submits a file, so a job can be one Go source file,
[`examples/wordcount.go`](wordcount.go), sent along with the job. `Map`
emits each word with a count of one, `Reduce` adds them up, and a single
line, `var Combine = api.Sum`, turns on the combiner. Each worker interprets
the file with Yaegi, a Go interpreter (ADR 0015), so nothing is compiled, no
binary is uploaded and the worker images never change.

The test is strict: the output must be byte for byte what the compiled
program wrote in section 4, and the cell asserts exactly that. Interpreting
does cost speed. E5 measured 2.3× slower than compiled for a file that, like
this one, calls the compiled helpers, and 10.2× without them. On a corpus
this small the difference is a fraction of a second.


In [ ]:
JOBFILE = f"/data/out/notebook-jobfile-{RUN}"
jobfile = submit("word count, job file", "-m", "4", "-r", "4", "examples/wordcount.go", JOBFILE, f"{CORPUS}/*.txt")
print(outcome(jobfile))
assert cat(JOBFILE) == output, "the job file and the compiled program differ"
passed(7, "The interpreted job file wrote the compiled program's output, byte for byte")


---

## 8. The combiner: fewer bytes shuffled, the same output

The combiner (§4.3) runs the job's `Combine` on each map task's sorted
output, so a word that occurs a thousand times in a split crosses the
network once, not a thousand times. Here the job file runs again with
`-combiner=false`: the output must not change, and `map.output.bytes`, the
bytes every reducer must fetch, shows what the combiner saves. The saving
grows with how often a word repeats within one map task's input: this
corpus is under 0.5 MB in four splits, so expect a few times fewer bytes
here. At 1 GiB the saving is 15× to 78× ([E1](../bench/README.md#evaluation-e0e7)).


That one `Combine` line deserves its own moment, because it is the paper's
§4.3 optimization. We run the same job file again with the combiner switched
off and compare. The number to watch is `map.output.bytes`, which is
everything the reducers have to fetch over the network.

With the combiner it is about 198 kB; without it, 606 kB, and Figure 2
puts the two side by side. That is 3.1 times
as much data for exactly the same answer, which the cell checks. The reason
is simple: a word that appears a thousand times in one split crosses the
network once instead of a thousand times. On half a megabyte both jobs
finish in under a second, too fast for the timing to mean much, but the
saving grows with the data. At 1 GiB it was 15× to 78× fewer bytes and
16–42% less job time (E1).


In [ ]:
NOCOMB = f"/data/out/notebook-nocombiner-{RUN}"
on = fields(jobfile)  # section 7's job, combiner on
off = fields(submit("word count, job file, no combiner", "-m", "4", "-r", "4", "-combiner=false", "examples/wordcount.go", NOCOMB, f"{CORPUS}/*.txt"))
assert cat(NOCOMB) == output, "the combiner changed the output"
saved = int(off["map.output.bytes"]) / int(on["map.output.bytes"])
assert saved > 1, f"the combiner saved no bytes: {on['map.output.bytes']} on, {off['map.output.bytes']} off"
for key in ("map.output.bytes", "reduce.input.records", "seconds"):
    print(f"{key:>22}  on {on[key]:>10}   off {off[key]:>10}")
passed(8, f"The combiner shuffled {saved:.1f}× fewer bytes, and the output did not change")

figure, axis = plt.subplots(figsize=(7, 2.2))
values = [int(on["map.output.bytes"]) / 1e3, int(off["map.output.bytes"]) / 1e3]
axis.barh(["combiner off", "combiner on"], values[::-1], color=[ORANGE, BLUE], height=0.55)
for y, v in enumerate(values[::-1]):
    axis.annotate(f"{v:,.0f} kB", (v, y), xytext=(4, 0), textcoords="offset points", va="center", color=INK2, fontsize=9)
axis.set_xlabel("Bytes shuffled to the reducers (kB)", color=INK2)
axis.set_xlim(0, max(values) * 1.15)
style(axis, f"The combiner shuffles {saved:.1f}× fewer bytes for the same answer")
figure.tight_layout()
plt.show()
caption(2, f"The bytes the map tasks wrote for the reducers to fetch: the job file of §7 with its combiner (blue) and again with `-combiner=false` (orange). Both runs wrote the same output; only the network traffic differs: {values[0]:,.0f} kB with the combiner against {values[1]:,.0f} kB without.")


---

## 9. A second program: the most requested pages


MapReduce is not only word count, so here is a second program from the
paper's §2.3 examples: URL access frequency, over a 10 MB web server log in
the Common Log Format. `Map` pulls the URL out of each request and `Reduce`
counts them. We ask for 2 map tasks and leave the reduce count at its
default of two per worker, which makes eight.

The log comes from a seeded generator that picks pages the way real traffic
does, with a long tail, so `/page1.html` gets about twice the hits of page 2
and three times those of page 3. This cell only checks that the job
succeeds. The counts are there to show a different kind of answer coming out
of the same machinery.


In [ ]:
assert (ROOT / "data/input/weblog/access.log").exists(), "run `make upload-example` first"
URLS = f"/data/out/notebook-urls-{RUN}"
print(outcome(submit("URL access frequency", "-m", "2", "urlfreq", URLS, "/data/input/weblog/access.log")))
urls = [line.split("\t") for line in cat(URLS).decode().splitlines()]
ranked = sorted(((url, int(n)) for url, n in urls), key=lambda pair: pair[1], reverse=True)
table(["Rank", "Page", "Requests"], [(rank, f"`{url}`", f"{count:,}") for rank, (url, count) in enumerate(ranked[:5], 1)])
passed(9, f"URL access frequency, a second program, succeeded: {len(ranked):,} distinct pages counted")


---

## 10. Kill a worker in the middle of a job

The coordinator notices a dead worker after a 10-second heartbeat timeout,
so this uses a job that lasts longer: a 1 GiB sort with 64 map tasks. Five
seconds in, while the job is still running, `docker kill` stops worker 4.
The job still ends `SUCCEEDED`, and the cell requires the coordinator's log
line that this worker, by the ID it registered with, was lost, and how many
of its tasks ran again elsewhere (§3.3 of the paper). The cell then restarts the worker and waits for all
four to be live.


Now the part that matters most in a distributed system: what happens when a
machine dies. The coordinator notices a dead worker only after ten seconds
without a heartbeat, so we need a job that lasts longer than that. We use a
1 GiB sort split into 64 map tasks of 16 MiB each, which takes about 13
seconds undisturbed on the course VM.

We start the sort, wait five seconds, make sure it is still running, and
then `docker kill` worker 4: no clean shutdown, as if someone pulled the
power cord. On the status page, that worker's last heartbeat starts climbing,
and at ten seconds the coordinator gives up on it and moves it to the *Lost
workers* table. Here is the subtle part, straight from the paper's §3.3. The
coordinator resets not only the task that worker was running but also the
map tasks it had already finished, because their output lived on its disk
and died with it. The map count on the status page drops back, and the
other three workers redo that work. Afterwards the cell restarts the
container, which registers as a brand-new worker under a new id.

The job still ends `SUCCEEDED`, with every one of its 10,737,419 records
written. The cell finds the coordinator's own log line, `worker lost …
tasks_reset=N`, naming the killed worker. N depends on how many map tasks
the worker had finished when it died; it has been between 8 and 17 in
rehearsals. The sort takes about 25 to 40 seconds instead of 13, and most of
the difference is the ten-second wait to be sure the worker is really gone,
plus redoing its work on three workers instead of four.


In [ ]:
SORT = f"/data/out/notebook-sort-kill-{RUN}"
if not KILL_WORKER:
    skipped(10, "A worker killed mid-job: the job still succeeds", "KILL_WORKER is False")
else:
    sort_input()
    names = containers()
    killed = next((w["id"] for w in status_json()["workers"] if names.get(w["host"]) == "mapreduce-worker-4"), None)
    assert killed, "mapreduce-worker-4 is not registered with the coordinator: check `docker compose ps`"
    reply = None
    job = subprocess.Popen(
        [MR, "submit", "-m", "64", "-r", "4", "-wait", "distsort", SORT, "/data/input/sort-1g.txt"],
        cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    try:
        time.sleep(5)
        assert job.poll() is None, "the sort finished within 5 s: no worker was lost mid-job"
        since = run("date", "-u", "+%Y-%m-%dT%H:%M:%SZ").stdout.strip()
        run("docker", "kill", "mapreduce-worker-4")
        print(f"killed mapreduce-worker-4, the coordinator's {killed}, five seconds into the job")
        reply, errors = job.communicate()
    finally:
        if reply is None:  # an error above: do not leave the client running
            job.kill()
        run(*COMPOSE, "up", "-d", "--no-build", "worker")
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    assert job.returncode == 0, errors
    status = fields(reply)
    JOBS[reply.split("\t")[0]] = "1 GiB sort, worker 4 killed"
    print(outcome(reply))
    assert reply.splitlines()[0].endswith("SUCCEEDED"), reply
    assert status["reduce.output.records"] == status["map.input.records"], "records lost"
    logs = run("docker", "logs", "--since", since, "mapreduce-coordinator-1")
    lost = [line for line in (logs.stdout + logs.stderr).splitlines() if "worker lost" in line and f"worker={killed} " in line]
    assert lost, f"the coordinator did not log losing {killed}, the killed worker"
    reset = int(lost[-1].split("tasks_reset=")[1].split()[0])
    assert reset > 0, lost[-1]
    print(lost[-1])
    passed(
        10,
        f"A worker killed mid-job: the coordinator lost {killed} and ran {reset} of its tasks again elsewhere,"
        f" and the job still wrote all {int(status['reduce.output.records']):,} records",
    )


---

## 11. Restart the coordinator in the middle of a job

The paper's master is a single point of failure: if it dies, the job is
aborted (§3.3). Here the coordinator writes every decision to a write-ahead
log before it replies ([ADR 0007](../docs/design/0007-write-ahead-log.md)),
so a restarted coordinator replays the log and finishes the job.

The same 1 GiB sort is submitted without `-wait`. Six seconds in, while the
job is still running, `docker kill` crashes the coordinator and
`docker start` brings it back, as `make e2e` does. (A `docker restart`
would stop it cleanly, and a clean stop tells idle workers to exit.) The job ends
`SUCCEEDED`, and its output is byte for byte the output of section 10's run.
On the course VM a restart costs about 1.2 s
([E4](../bench/README.md#evaluation-e0e7)).


Losing a worker is something the paper handles. Losing the master is not: in
2004, if the master died, the job was aborted and the client had to start
over. We went further. Our coordinator writes every decision to a write-ahead
log on disk before it replies to anyone (ADR 0007), so a restarted
coordinator can replay the log and carry on exactly where it left off.

So we submit the same sort again, this time without waiting on it, because
the client's connection would die along with the coordinator. Six seconds
in, the cell confirms the job is still mapping or reducing, then crashes the
coordinator with `docker kill` and brings it straight back with `docker
start`. For a few seconds nothing answers: the status page stops responding,
and the cell simply keeps polling. Then the page comes back with the same
job still in progress and every earlier job from this run still listed, all
rebuilt from the log. The workers re-register under new ids, `r2-w1` to
`r2-w4`, because this is the coordinator's second life.

The job ends `SUCCEEDED` within about 20 seconds, and the last line is the
one that matters: this run's output is byte for byte identical to section
10's. One sort lost a worker, the other lost its coordinator, and both wrote
exactly the same gigabyte. In the evaluation (E4), a restart cost about 1.2
seconds and kept every finished task.


In [ ]:
RESTARTED = f"/data/out/notebook-sort-restart-{RUN}"
if not RESTART_COORDINATOR:
    skipped(11, "The coordinator crashed mid-job: the job resumes from its log", "RESTART_COORDINATOR is False")
else:
    sort_input()
    started = time.monotonic()
    job_id = run(MR, "submit", "-m", "64", "-r", "4", "distsort", RESTARTED, "/data/input/sort-1g.txt").stdout.split()[0]
    time.sleep(6)
    before = run(MR, "status", job_id, timeout=10).stdout.splitlines()[0]
    print(before, "<- before the restart")
    assert before.split("\t")[1] in ("PENDING", "MAPPING", "REDUCING"), "the job ended before the restart"
    run("docker", "kill", "mapreduce-coordinator-1")
    run("docker", "start", "mapreduce-coordinator-1")
    print("crashed and started mapreduce-coordinator-1")
    state, reply = "", None
    while time.monotonic() - started < 300:
        try:  # Unavailable while it restarts; a hung request counts as no answer
            reply = run(MR, "status", job_id, check=False, timeout=10)
        except subprocess.TimeoutExpired:
            reply = None
        state = reply.stdout.split("\t", 1)[1].split("\n", 1)[0] if reply and reply.returncode == 0 else ""
        if state in ("SUCCEEDED", "FAILED"):
            break
        time.sleep(1)
    JOBS[job_id] = "1 GiB sort, coordinator restarted"
    print(f"{job_id} {state} after {time.monotonic() - started:.0f} s")
    assert state == "SUCCEEDED", (reply.stdout + reply.stderr) if reply else "no status reply"
    assert wait_for_workers(), "fewer than 4 live workers after 60 s: check `docker compose ps`"
    passed(11, "The coordinator crashed mid-job: the job resumed from its log and succeeded")
    if KILL_WORKER:
        assert digest(RESTARTED) == digest(SORT), "the restarted job's output differs"
        passed(11, "Both fault runs wrote the same 1 GiB of output, byte for byte")
    else:
        skipped(11, "Both fault runs wrote the same 1 GiB of output, byte for byte", "KILL_WORKER is False")


---

## 12. The coordinator's metrics

Every job this run submitted, as the coordinator reports it on its status
page (state, tasks, time and backup tasks, §3.6), then the raw Prometheus
metrics the status page and the Grafana dashboard are drawn from: live
workers, and completed tasks per job.


Now we look at what the coordinator itself recorded. This table lists every
job this notebook submitted, read from the status page's data, beside the
label we gave it. All six succeeded (fewer appear if a section was skipped),
and the backups column shows the paper's backup tasks turning up even in
small jobs, for the reason we saw in section 4. Compare the two sorts:
losing a worker costs the ten-second timeout plus the redone work, while a
coordinator restart costs about a second, so the sort with the killed worker
is the slower of the two.

Every job from before section 11 is still here, and that is the point: this
list survived the coordinator crash because it was rebuilt from the log.
Below the table are the raw Prometheus metrics that the status page and the
Grafana dashboard are drawn from: the live workers, and the completed tasks
of each job.


In [ ]:
jobs = {job["id"]: job for job in status_json()["jobs"]}
table(
    ["Job", "What it ran", "State", "Map tasks", "Reduce tasks", "Seconds", "Backups"],
    [
        (
            job,
            label,
            jobs[job]["state"],
            f'{jobs[job]["map_completed"]}/{jobs[job]["map_total"]}',
            f'{jobs[job]["reduce_completed"]}/{jobs[job]["reduce_total"]}',
            f'{jobs[job]["duration_seconds"]:.1f}',
            jobs[job]["backup_attempts"],
        )
        for job, label in JOBS.items()
        if job in jobs
    ],
)
print("The raw metrics, from the coordinator's /metrics endpoint:")
for line in metrics_lines():
    if line.startswith("mr_workers ") or ('state="completed"' in line and any(f'job="{job}"' in line for job in JOBS)):
        print(line)

---

## 13. Local mode is the oracle

Local mode runs the same job in one process, without Docker. Its output, the
cluster's and the expected output checked in at `testdata/corpus/expected`
must all be the same, byte for byte (README, "Run a job on one machine").


Finishing a job is not the same as getting the right answer, so here is how
we know the answers are right. `mr run -local` runs the same word count in a
single process on the host, with no Docker, no network and no coordinator.
That is the paper's §4.7 local mode, and it serves as our oracle. Three
outputs must then match byte for byte: the cluster's from section 4, local
mode's, and the expected output committed in `testdata/corpus/expected/`.
They do. The Python implementation writes those same bytes too, and
`make e2e` and CI check this on every change, not just today.


In [ ]:
LOCAL = f"data/out/notebook-local-{RUN}"
run(MR, "run", "-local", "-m", "4", "-r", "4", "wordcount", LOCAL, "testdata/corpus/*.txt")
assert cat(f"/{LOCAL}") == output, "local mode and the cluster differ"
passed(13, "Local mode and the cluster wrote the same output, byte for byte")
assert output == EXPECTED, "the cluster's output differs from testdata/corpus/expected"
passed(13, "The cluster's output matches the committed expected output, byte for byte")


---

## 14. The evaluation at a glance

The paper's §5 experiments and our own, rerun on the course VM with 1 GiB
inputs: medians of five runs, every run's output checked against an
undisturbed run's. `bench/plot-experiments.py` draws them from the data in
`bench/evaluation/`, in order E0 to E7, with the fault recap after E4; [bench/README.md](../bench/README.md#evaluation-e0e7)
has every figure, table and command.


Everything so far ran on a small corpus or a single sort. These figures show
what the system does at scale, measured on the course VM with 1 GiB inputs.
Each number is the median of five runs with the settings taking turns, and
every run's output was checked against an undisturbed run's. Nothing is
rerun here, since the full set of experiments takes about two hours
(`bench/experiments.sh`), so the cell shows the committed figures, each
captioned with its finding.

Three are worth slowing down for. E0 shows word count running 3.7× faster on
four workers than on one, so the work really does spread. E1 is section 8's
combiner saving at scale. The faults summary shows what sections 10 and 11
cost, measured five times each.

There is one fair caveat. Four one-CPU workers on one machine, sharing one
disk, are not 1,800 machines. Fixed costs such as startup, heartbeats and
the ten-second failure timeout are a large share of an 8-second job, which
is why a lost worker costs proportionally more here than in the paper
([evaluation](../docs/evaluation.md)). If the projector needs it,
`THEME = "-dark"` shows the dark versions.


In [ ]:
THEME = ""  # "-dark" for the dark variants
# Each figure's finding, as README.md's results table states it.
FIGURES = {
    "E0": "E0, scaling over workers: word count runs 3.7× faster on 4 workers than on 1, distributed sort 3.1×, grep 2.0×.",
    "E1": "E1, the combiner: 15× to 78× fewer bytes shuffled and 16–42% less job time, more as the keys get more skewed.",
    "E2": "E2, backup tasks: with one worker at 0.2 CPU, backups cut 17.9 s to 11.4 s; with no straggler they cost nothing measurable.",
    "E3": "E3, losing a worker: every job finishes with the undisturbed run's output, +8 s to +26 s in all.",
    "E4": "E4, a coordinator restart: recovery from the log costs 1.2 s (15%) and keeps every finished task.",
    "faults": "E2 to E4 together: what each fault costs.",
    "E5": "E5, an interpreted job file: 2.3× slower than compiled with the helpers api.Words/api.Sum, 10.2× without them.",
    "E6": "E6, the combiner on real text: 5.2× fewer bytes and 11% less job time on crawled English.",
    "E7": "E7, the paper's backups against LATE: either rule cuts a sort on mixed-speed workers by 32%; LATE needs fewer backups, and the two times overlap.",
}
for number, (name, finding) in enumerate(FIGURES.items(), start=3):  # figures 1 and 2 are §6's and §8's
    display(Image(filename=str(ROOT / "bench/evaluation" / f"{name}{THEME}.png"), width=760))
    caption(number, finding)

That is the whole story. Each of its claims was checked by an `assert` as
it ran, and the scorecard below gathers them: every claim that held on this
cluster today, and any section switched off in the setup cell, marked as not
tested. Two small functions became a distributed job across four workers,
whether compiled in or sent as a plain Go file. The combiner cut the network
traffic without changing a byte of the answer. With both fault sections
switched on, a worker killed mid-job and a coordinator crashed mid-job both
ended in success with identical output, and the second goes beyond what the
2004 paper could do; the scorecard shows whether they ran this time. The
cluster, local mode and the committed answer agree byte for byte. The
evaluation figures in section 14 are earlier measurements, not this run's.

Section 15 is there if something goes wrong, and section 16 tidies up this
run's files.


In [ ]:
# Every claim this run checked, in section order. A section switched off in
# the setup cell shows as skipped, so the table never claims more than ran.
rows = sorted(CHECKS.items(), key=lambda item: item[0][0])  # by section, then in the order checked
table(["§", "Claim", "Result"], [(section, claim, result) for (section, _), (claim, result) in rows])
held = sum(result.startswith("✓") for _, result in CHECKS.values())
print(f"{held} of {len(CHECKS)} claims held on this cluster today.")

---

## 15. If a cell fails

Every check stops with a message that names its cause. Rerunning from the
top is safe once no other copy is running: setup clears what an earlier or
interrupted run left, including directories a running copy still uses.

| Message or symptom | What to run |
|---|---|
| `bin/mr is missing`, `not answering`, `no live workers` | `make cluster-up mr`, then `docker compose -p mapreduce -f deploy/compose/compose.yaml ps` |
| `fewer than 4 live workers after 60 s` | `docker compose -p mapreduce -f deploy/compose/compose.yaml up -d --no-build worker` |
| ``run `make upload-example` first`` | `make upload-example` |
| `GiB free: the sort needs 3 GiB` | free disk space, or set `KILL_WORKER` and `RESTART_COORDINATOR` to `False` |
| `the sort finished within 5 s`, `the job ended before the restart` | the machine is faster than the course VM: rerun the cell, or skip it with its flag |
| Anything else | `make demo-check`: one `OK`, `WARN` or `FAIL` line per thing the demo needs, each `FAIL` with what to run |


---

## 16. Clean up

This run's input and output. The sort input stays in `data/input/` for the next run.


In [ ]:
# A section that did not run leaves its name undefined, so clean up only
# the paths the notebook actually created (the setup always defines CORPUS).
for name in ("WC", "JOBFILE", "NOCOMB", "URLS", "SORT", "RESTARTED", "LOCAL", "CORPUS"):
    path = globals().get(name)
    if path:
        shutil.rmtree(ROOT / path.removeprefix("/"), ignore_errors=True)
print("removed this run's input and output")
